In [0]:
# Importar bibliotecas necessárias
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pyspark.sql.window import Window

# Carregar a tabela tratada
df_covid = spark.table("covid_data_tratado")

print("Análise da base PNAD-COVID-19")
print("=" * 50)

# Verificar os meses disponíveis na base
meses_disponiveis = df_covid.select("MES_PESQUISA").distinct().orderBy("MES_PESQUISA").collect()
meses = [row["MES_PESQUISA"] for row in meses_disponiveis]
print(f"Meses disponíveis na base: {meses}")

# Selecionar 3 meses para análise (conforme solicitado)
meses_selecionados = meses[:3] if len(meses) >= 3 else meses
print(f"Meses selecionados para análise: {meses_selecionados}")

# Filtrar para os 3 meses selecionados
df_analise = df_covid.filter(F.col("MES_PESQUISA").isin(meses_selecionados))

# Verificar contagem de registros após filtro
print(f"Total de registros para análise: {df_analise.count():,}")

# ===== PARTE 1: CARACTERÍSTICAS CLÍNICAS DOS SINTOMAS =====
print("\n1. CARACTERÍSTICAS CLÍNICAS DOS SINTOMAS")
print("-" * 50)

# Colunas de sintomas
sintomas_cols = [col for col in df_analise.columns if col.startswith("SE_TEVE_")]

# Prevalência de sintomas
print("Prevalência de sintomas na população:")
for col in sintomas_cols:
    sintoma_nome = col.replace("SE_TEVE_", "")
    contagem = df_analise.filter(F.col(col) == "Sim").count()
    percentual = (contagem / df_analise.count()) * 100
    print(f"- {sintoma_nome}: {percentual:.2f}% ({contagem:,} pessoas)")

# Sintomas por faixa etária
print("\nPrevalência de sintomas graves por faixa etária:")
sintomas_graves_faixa = df_analise.filter(F.col("SINTOMAS_GRAVES") == "Sim") \
                        .groupBy("FAIXA_ETARIA") \
                        .count() \
                        .withColumn("percentual", F.col("count") / df_analise.count() * 100) \
                        .orderBy("count", ascending=False)
display(sintomas_graves_faixa)

# Busca por atendimento médico
print("\nBusca por atendimento médico entre pessoas com sintomas:")
atendimento_medico = df_analise.filter(F.col("SINTOMAS_GRAVES") == "Sim") \
                    .groupBy("FOI_AO_MEDICO") \
                    .count() \
                    .withColumn("percentual", F.col("count") / df_analise.filter(F.col("SINTOMAS_GRAVES") == "Sim").count() * 100) \
                    .orderBy("count", ascending=False)
display(atendimento_medico)

Análise da base PNAD-COVID-19
Meses disponíveis na base: [5, 6, 7]
Meses selecionados para análise: [5, 6, 7]
Total de registros para análise: 1,114,742

1. CARACTERÍSTICAS CLÍNICAS DOS SINTOMAS
--------------------------------------------------
Prevalência de sintomas na população:
- FEBRE: 1.87% (20,825 pessoas)
- TOSSE: 2.65% (29,554 pessoas)
- DOR_GARGANTA: 2.04% (22,769 pessoas)
- FALTA_AR: 1.06% (11,858 pessoas)
- DOR_CABECA: 3.85% (42,936 pessoas)
- DOR_PEITO: 0.86% (9,623 pessoas)
- NAUSEA: 0.83% (9,284 pessoas)
- NARIZ_ENTUPIDO: 2.92% (32,592 pessoas)
- FADIGA: 1.35% (15,067 pessoas)
- DOR_OLHOS: 1.01% (11,206 pessoas)
- PERDA_PALADAR: 1.25% (13,946 pessoas)
- DOR_MUSCULAR: 2.27% (25,318 pessoas)

Prevalência de sintomas graves por faixa etária:


FAIXA_ETARIA,count,percentual
Adulto,2834,0.2542292297231108
Idoso,581,0.0521196832989158
Jovem,517,0.046378444519000805
Adolescente,276,0.02475909223838341
Criança,220,0.01973550830595779



Busca por atendimento médico entre pessoas com sintomas:


FOI_AO_MEDICO,count,percentual
Sim,2579,58.242999096657634
Não,1843,41.62149954832881
Ignorado,6,0.13550135501355012


In [0]:
# ===== PARTE 2: CARACTERÍSTICAS DA POPULAÇÃO =====
print("\n2. CARACTERÍSTICAS DA POPULAÇÃO")
print("-" * 50)

# Distribuição por região
print("Distribuição da população por região:")
regiao_count = df_analise.groupBy("REGIAO") \
              .count() \
              .withColumn("percentual", F.col("count") / df_analise.count() * 100) \
              .orderBy("count", ascending=False)
display(regiao_count)

# Distribuição por faixa etária
print("\nDistribuição por faixa etária:")
faixa_etaria_count = df_analise.groupBy("FAIXA_ETARIA") \
                    .count() \
                    .withColumn("percentual", F.col("count") / df_analise.count() * 100) \
                    .orderBy("count", ascending=False)
display(faixa_etaria_count)

# Distribuição por nível de risco
print("\nDistribuição por nível de risco:")
risco_count = df_analise.groupBy("NIVEL_RISCO") \
             .count() \
             .withColumn("percentual", F.col("count") / df_analise.count() * 100) \
             .orderBy("count", ascending=False)
display(risco_count)

# Vulnerabilidade por região
print("\nPopulação vulnerável por região:")
vulneravel_regiao = df_analise.filter(F.col("VULNERAVEL") == "Sim") \
                   .groupBy("REGIAO") \
                   .count() \
                   .withColumn("percentual", F.col("count") / df_analise.filter(F.col("VULNERAVEL") == "Sim").count() * 100) \
                   .orderBy("count", ascending=False)
display(vulneravel_regiao)


2. CARACTERÍSTICAS DA POPULAÇÃO
--------------------------------------------------
Distribuição da população por região:


REGIAO,count,percentual
Nordeste,336264,30.165186204520865
Sudeste,332210,29.801514610555625
Sul,192162,17.23824885040664
Norte,135435,12.149448033715425
Centro-Oeste,118671,10.64560230080144



Distribuição por faixa etária:


FAIXA_ETARIA,count,percentual
Adulto,544821,48.874178958001046
Idoso,191001,17.134099190664745
Criança,164630,14.76843969277196
Jovem,114334,10.256543666606264
Adolescente,99956,8.966738491955986



Distribuição por nível de risco:


NIVEL_RISCO,count,percentual
Baixo,919894,82.5207985345488
Alto,134830,12.095175385874041
Médio,59437,5.331906396278241
Muito Alto,581,0.0521196832989158



População vulnerável por região:


REGIAO,count,percentual
Sudeste,41426,30.6212809993717
Nordeste,41083,30.367742173929113
Sul,26243,19.398307277229552
Norte,13681,10.11272498798832
Centro-Oeste,12852,9.499944561481318


In [0]:
# ===== PARTE 3: CARACTERÍSTICAS ECONÔMICAS DA SOCIEDADE =====
print("\n3. CARACTERÍSTICAS ECONÔMICAS DA SOCIEDADE")
print("-" * 50)

# Auxílio emergencial por região
print("Recebimento de auxílio emergencial por região:")
auxilio_regiao = df_analise.filter(F.col("AUXILIO_EMERGENCIAL_COVID") == "Sim") \
                .groupBy("REGIAO") \
                .count() \
                .withColumn("percentual", F.col("count") / df_analise.filter(F.col("AUXILIO_EMERGENCIAL_COVID") == "Sim").count() * 100) \
                .orderBy("count", ascending=False)
display(auxilio_regiao)

# Trabalho remoto por escolaridade
print("\nTrabalho remoto por nível de escolaridade:")
if "TRABALHO_REMOTO" in df_analise.columns and "ESCOLARIDADE" in df_analise.columns:
    remoto_escolaridade = df_analise.filter(F.col("TRABALHO_REMOTO") == "Sim") \
                         .groupBy("ESCOLARIDADE") \
                         .count() \
                         .withColumn("percentual", F.col("count") / df_analise.filter(F.col("TRABALHO_REMOTO") == "Sim").count() * 100) \
                         .orderBy("count", ascending=False)
    display(remoto_escolaridade)


3. CARACTERÍSTICAS ECONÔMICAS DA SOCIEDADE
--------------------------------------------------
Recebimento de auxílio emergencial por região:


REGIAO,count,percentual
Nordeste,219878,39.68580282900187
Sudeste,138758,25.04444568782071
Norte,83000,14.980678534492561
Sul,60383,10.898533878894751
Centro-Oeste,52028,9.390539069790108



Trabalho remoto por nível de escolaridade:


ESCOLARIDADE,count,percentual
Superior completo,18218,48.83527677255059
"Pós-graduação, mestrado ou doutorado",9342,25.042219541616404
Médio completo,4640,12.43801099048385
Superior incompleto,4029,10.800160836349015
Médio incompleto,443,1.187508376893178
Fundamental incompleto,322,0.8631550730465086
Fundamental completa,295,0.7907787159898138
Sem instrução,16,0.042889693070633964


In [0]:
# ===== PARTE 4: COMPORTAMENTO DA POPULAÇÃO NA ÉPOCA DA COVID-19 =====
print("\n4. COMPORTAMENTO DA POPULAÇÃO NA ÉPOCA DA COVID-19")
print("-" * 50)

# Análise de isolamento social
if "ISOLAMENTO_EFETIVO" in df_analise.columns:
    print("Nível de isolamento social:")
    isolamento_count = df_analise.groupBy("ISOLAMENTO_EFETIVO") \
                      .count() \
                      .withColumn("percentual", F.col("count") / df_analise.count() * 100) \
                      .orderBy("count", ascending=False)
    display(isolamento_count)

# Relação entre isolamento e sintomas graves
if "ISOLAMENTO_EFETIVO" in df_analise.columns and "SINTOMAS_GRAVES" in df_analise.columns:
    print("\nRelação entre isolamento social e sintomas graves:")
    isolamento_sintomas = df_analise.groupBy("ISOLAMENTO_EFETIVO", "SINTOMAS_GRAVES") \
                         .count() \
                         .orderBy("ISOLAMENTO_EFETIVO", "SINTOMAS_GRAVES")
    display(isolamento_sintomas)


4. COMPORTAMENTO DA POPULAÇÃO NA ÉPOCA DA COVID-19
--------------------------------------------------


In [0]:
# ===== PARTE 5: RECOMENDAÇÕES PARA NOVO SURTO =====
print("\n5. RECOMENDAÇÕES PARA NOVO SURTO DE COVID-19")
print("-" * 50)

# Identificar grupos de maior risco
print("Grupos de maior risco identificados:")
grupos_risco = df_analise.filter(F.col("NIVEL_RISCO").isin(["Muito Alto", "Alto"])) \
              .groupBy("FAIXA_ETARIA", "REGIAO") \
              .count() \
              .orderBy("count", ascending=False) \
              .limit(5)
display(grupos_risco)

# Regiões com maior incidência de sintomas graves
print("\nRegiões com maior incidência de sintomas graves:")
regioes_sintomas = df_analise.filter(F.col("SINTOMAS_GRAVES") == "Sim") \
                  .groupBy("REGIAO") \
                  .count() \
                  .withColumn("percentual", F.col("count") / df_analise.filter(F.col("SINTOMAS_GRAVES") == "Sim").count() * 100) \
                  .orderBy("count", ascending=False)
display(regioes_sintomas)

# Análise de eficácia do isolamento
if "ISOLAMENTO_EFETIVO" in df_analise.columns and "SINTOMAS_GRAVES" in df_analise.columns:
    print("\nEficácia do isolamento na redução de sintomas graves:")

    # Calcular taxa de sintomas graves por nível de isolamento
    isolamento_eficacia = df_analise.groupBy("ISOLAMENTO_EFETIVO") \
                         .agg(
                             F.count("*").alias("total"),
                             F.sum(F.when(F.col("SINTOMAS_GRAVES") == "Sim", 1).otherwise(0)).alias("casos_graves")
                         ) \
                         .withColumn("taxa_sintomas_graves", F.col("casos_graves") / F.col("total") * 100) \
                         .orderBy("taxa_sintomas_graves")
    display(isolamento_eficacia)

# Resumo das principais conclusões
print("\nPRINCIPAIS CONCLUSÕES E RECOMENDAÇÕES:")
print("""
1. Focar atendimento em grupos de maior risco (idosos, especialmente sem plano de saúde)
2. Priorizar regiões com maior incidência de casos graves
3. Promover isolamento social efetivo, especialmente para grupos vulneráveis
4. Garantir acesso a atendimento médico para pessoas com sintomas
5. Considerar suporte econômico para populações vulneráveis
""")

# Salvar resultados da análise em um novo DataFrame para referência
analise_resultados = {
    "sintomas_por_faixa": sintomas_graves_faixa,
    "atendimento_medico": atendimento_medico,
    "distribuicao_regional": regiao_count,
    "distribuicao_faixa_etaria": faixa_etaria_count,
    "niveis_risco": risco_count,
    "vulnerabilidade_regional": vulneravel_regiao,
    "auxilio_emergencial": auxilio_regiao,
    "grupos_maior_risco": grupos_risco,
    "regioes_maior_incidencia": regioes_sintomas
}

# Criar um DataFrame com os resultados principais
resultados_df = spark.createDataFrame([
    ("Sintomas mais comuns", "Febre, Tosse, Dor de Cabeça"),
    ("Grupos de maior risco", "Idosos sem plano de saúde"),
    ("Regiões mais afetadas", "A ser determinado pela análise"),
    ("Eficácia do isolamento", "A ser determinado pela análise"),
    ("Impacto econômico", "Maior em regiões com menor escolaridade")
], ["Categoria", "Resultado"])

# Salvar resultados
resultados_df.write.format("delta").mode("overwrite").saveAsTable("covid_analise_resultados")
print("\nResultados da análise salvos na tabela 'covid_analise_resultados'")


5. RECOMENDAÇÕES PARA NOVO SURTO DE COVID-19
--------------------------------------------------
Grupos de maior risco identificados:


FAIXA_ETARIA,REGIAO,count
Idoso,Sudeste,41470
Idoso,Nordeste,41115
Idoso,Sul,26252
Idoso,Norte,13706
Idoso,Centro-Oeste,12868



Regiões com maior incidência de sintomas graves:


REGIAO,count,percentual
Norte,1710,38.61788617886179
Nordeste,1226,27.68744354110208
Sudeste,855,19.308943089430894
Centro-Oeste,390,8.807588075880759
Sul,247,5.578139114724481



PRINCIPAIS CONCLUSÕES E RECOMENDAÇÕES:

1. Focar atendimento em grupos de maior risco (idosos, especialmente sem plano de saúde)
2. Priorizar regiões com maior incidência de casos graves
3. Promover isolamento social efetivo, especialmente para grupos vulneráveis
4. Garantir acesso a atendimento médico para pessoas com sintomas
5. Considerar suporte econômico para populações vulneráveis


Resultados da análise salvos na tabela 'covid_analise_resultados'
